# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library. All entities (record sets, fields, columns, etc.) are referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")print(f"Authors: {getattr(metadata, 'author', None)}")print(f"Keywords: {getattr(metadata, 'keywords', None)}")print(f"Published on: {getattr(metadata, 'datePublished', None)}")print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore record sets in the metadata
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets listed in top-level metadata, attempting auto-discovery…')
    # mlcroissant auto-discovers record sets if not listed in metadata
    # so we try reading one record to enumerate
    all_record_sets = dataset.list_record_sets()
    print(f"Discovered record sets (by @id): {all_record_sets}")
    record_sets = all_record_sets
else:
    # In case record_sets are present, extract their @id
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]
    print(f"Record sets in dataset (by @id): {record_sets}")

# For each record set, list its fields (columns)
for record_set_id in record_sets:
    print(f"\nRecord set: {record_set_id}")
    info = dataset.record_set_info(record_set_id)
    if info is not None:
        fields = info.get('fields', [])
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '')} (type: {field.get('dataType', '')})")
    else:
        print("  Could not find fields information.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List of record set @id's (discovered above)
record_set_ids = record_sets
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}")
        print(f"Columns (@id): {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
    else:
        print(f"No records found in record set {record_set_id}")
# Select a main record set for further EDA
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Use the first as default
    print(f"\nDefaulting to main record set: {main_record_set_id}, which has columns:\n{dataframes[main_record_set_id].columns.values}")
else:
    print("No extracted dataframes. Please check the Croissant schema/dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes for further analysis.

In [ ]:
import numpy as np

# For demonstration, attempt to select a numeric field (@id) for EDA
df = dataframes[main_record_set_id]
numeric_field = None
group_field = None

# Try to find a likely numeric field and a grouping field based on DataFrame dtypes and names
for col in df.columns:
    if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
        if numeric_field is None:
            numeric_field = col
    elif df[col].dtype == object:
        if group_field is None and df[col].nunique() < df.shape[0] // 3:
            group_field = col

print(f"Selected numeric_field: {numeric_field}")
if group_field:
    print(f"Selected group_field for grouping: {group_field}")

# If we have a numeric field, try filtering:
if numeric_field:
    # By default, threshold is the median
    threshold = df[numeric_field].median() if np.issubdtype(df[numeric_field].dtype, np.number) else None
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (total: {len(filtered_df)}):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field exists, boxplot by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we explored and analyzed the FAIR<sup>2</sup> mass spectrometry protein dataset using `mlcroissant`:
- Loaded Croissant metadata and reviewed dataset structure.
- Enumerated record sets and fields using their `@id`s.
- Loaded records and demonstrated EDA on key numeric columns, with filtering and normalization.
- Visualized distributions and compared groups where available.

**Note:** For deeper analysis, please refer to the detailed documentation and consider the biological context of the fields for advanced biomarker or protein expression studies.